In [2]:
# ==============================================================
# 纯净版消融实验：双实验设置 (Random Split vs. Temporal Transfer)
# 综合对比测试：7 种 SOTA MOEAs (包含 MOFS-REPLS 和 DREA-FS 专版)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy openpyxl
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration & Utilities
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (基于随机森林特征重要性)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Class_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_class'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Min_Year_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return float(np.mean(opts['imp_year'][sel])) + _size_penalty(len(sel), len(x))

# ==============================================================
# 2. Pymoo Problem Definition
# ==============================================================
class FeatureSelectionProblem(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Class_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Min_Year_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 3. 算法核心算子定义 (Samplings & Algorithms)
# ==============================================================
class REPLSSampling(Sampling):
    """针对 MOFS-REPLS 算法定制的偏好注入采样器"""
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        weights = problem.opts['imp_class']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    """MOFS-REPLS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0]
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                p_nearest_indices = np.argsort(dists)[:self.P]
                
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0: 
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66: 
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False
            infills.set("X", X_inf)
        return infills


class DREAFSSampling(Sampling):
    """针对 DREA-FS 算法定制的双视角降维采样器"""
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_class = problem.opts['imp_class']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_class[imp_class > 0]) if np.any(imp_class > 0) else 0
        important_features = np.where(imp_class >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
        return X_init

class DREA_FS(NSGA2):
    """DREA-FS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 4. Core Evaluation Routine (7 种算法大一统)
# ==============================================================
def run_moeas_and_evaluate(xtrain, xtest, ytrain, ytest, yyear_tr, run_idx, exp_name):
    print(f"  [{exp_name} - Run {run_idx}] Pre-computing Feature Importances (Random Forest)...")
    
    rf_class = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf_class.fit(xtrain, ytrain)
    imp_class = rf_class.feature_importances_
    
    rf_year = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf_year.fit(xtrain, yyear_tr)
    imp_year = rf_year.feature_importances_
    
    opts = {'xt_in': xtrain, 'imp_class': imp_class, 'imp_year': imp_year, 'ytrain': ytrain}
    problem = FeatureSelectionProblem(opts)
    
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
    pop_size = len(ref_dirs) # 91
    
    # 统一算子
    crossover = SinglePointCrossover(prob=0.8)
    mutation = BitflipMutation(prob=0.01)
    std_sampling = BinaryRandomSampling()
    
    # 将 7 种算法全部放进同一个迭代字典
    algorithms = {
        "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
        "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
    }
    
    def get_top_k(scores, k):
        idx = np.argsort(scores)[::-1][:k]
        return idx if len(idx) > 0 else np.array([0])
        
    clf = LinearDiscriminantAnalysis()
    results = []
    
    for algo_name, algo in algorithms.items():
        print(f"    -> Optimizing with {algo_name}...")
        
        # --- 核心修改：记录运算耗时 ---
        start_time = time.time()
        res = minimize(problem, algo, ('n_gen', 40), seed=42+run_idx, verbose=False)
        elapsed_time = time.time() - start_time
        # ----------------------------
        
        final_pop = res.pop.get("X").astype(int)
        scores_algo = final_pop.mean(axis=0)
        
        for k in range(1, 16):
            idx = get_top_k(scores_algo, k)
            clf.fit(xtrain[:, idx], ytrain)
            acc = clf.score(xtest[:, idx], ytest)
            
            results.append({
                "Experiment": exp_name,
                "Run": run_idx, 
                "Method": algo_name, 
                "K": k, 
                "Accuracy": acc,
                "Time_Seconds": elapsed_time # 将对应算法的整体运行时间存入 DataFrame
            })
            
    return results

# ==============================================================
# 5. Dual-Experiment Orchestrator
# ==============================================================
def run_sota_benchmark(features, class_labels, year_raw, target_year_val=4, n_runs=10):
    le_year = LabelEncoder()
    y_year_encoded = le_year.fit_transform(year_raw)
    
    if target_year_val in year_raw:
        target_year_idx = le_year.transform([target_year_val])[0]
    else:
        target_year_idx = np.max(y_year_encoded)
        print(f"Warning: target_year_val {target_year_val} not found, using encoded year {target_year_idx} for testing.")

    all_results = []
    print("="*70)
    print(f"Starting UNIFIED MOEA Benchmark: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Starting Run {run_idx}/{n_runs}...")
        
        # --- Exp 1: Random Split ---
        xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, _ = train_test_split(
            features, class_labels, y_year_encoded, 
            test_size=0.3, stratify=class_labels, random_state=current_seed
        )
        res1 = run_moeas_and_evaluate(xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, run_idx, "Exp1_Random_Split")
        all_results.extend(res1)
        
        # --- Exp 2: Temporal Transfer ---
        test_mask = (y_year_encoded == target_year_idx)
        train_mask = ~test_mask
        xt_tr2, xt_te2 = features[train_mask], features[test_mask]
        yt_tr2, yt_te2 = class_labels[train_mask], class_labels[test_mask]
        yr_tr2 = y_year_encoded[train_mask]
        
        res2 = run_moeas_and_evaluate(xt_tr2, xt_te2, yt_tr2, yt_te2, yr_tr2, run_idx, "Exp2_Temporal_Transfer")
        all_results.extend(res2)

    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("             UNIFIED MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"
    
    # ----------------------------------------------------
    # 表 1. 分类精确度 (Experiment 1: Random Split)
    # ----------------------------------------------------
    df_exp1 = df_results[df_results["Experiment"] == "Exp1_Random_Split"]
    table_acc_exp1 = df_exp1.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp1 = table_acc_exp1[[m for m in METHOD_ORDER if m in table_acc_exp1.columns]]
    
    print("\n[TABLE 1] Exp 1 (Random Split) - Classification Accuracy")
    print(table_acc_exp1)
    table_acc_exp1.to_csv("Unified_Exp1_Accuracy.csv") 
    
    # ----------------------------------------------------
    # 表 2. 分类精确度 (Experiment 2: Temporal Transfer)
    # ----------------------------------------------------
    df_exp2 = df_results[df_results["Experiment"] == "Exp2_Temporal_Transfer"]
    table_acc_exp2 = df_exp2.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp2 = table_acc_exp2[[m for m in METHOD_ORDER if m in table_acc_exp2.columns]]
    
    print("\n[TABLE 2] Exp 2 (Temporal Transfer) - Classification Accuracy")
    print(table_acc_exp2)
    table_acc_exp2.to_csv("Unified_Exp2_Accuracy.csv")

    # ----------------------------------------------------
    # 表 3. 计算耗时对比 (Time_Seconds)
    # 我们按方法进行 GroupBy，合并两种实验模式下的平均耗时
    # ----------------------------------------------------
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Unified_Execution_Time.csv")

    print("\n-> 大一统 7 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/Spectral_DataSet(1).xlsx")
        
        class_col_idx = -4
        year_col_idx = -1
        y_class_raw = df.iloc[:, class_col_idx].values
        mask_cls = np.isin(y_class_raw, [2, 3])
        df_filtered = df[mask_cls].copy()
        
        balanced_indices = []
        curr_years = df_filtered.iloc[:, year_col_idx].values
        curr_classes = df_filtered.iloc[:, class_col_idx].values
        unique_years = np.unique(curr_years)
        
        for yr in unique_years:
            for cls in [2, 3]:
                match_mask = (curr_years == yr) & (curr_classes == cls)
                match_indices = np.where(match_mask)[0]
                if len(match_indices) >= 100:
                    selected = np.random.choice(match_indices, 100, replace=False)
                else: 
                    selected = match_indices
                balanced_indices.extend(selected)
                
        df_balanced = df_filtered.iloc[balanced_indices].copy()
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X = df_balanced.iloc[:, :600].values.astype(float)
        X = normalize(X + 1e-8, norm='l1', axis=1)
        y_cls = np.where(df_balanced.iloc[:, class_col_idx].values == 2, 0, 1)
        y_year_raw = df_balanced.iloc[:, year_col_idx].values
        
        run_sota_benchmark(X, y_cls, y_year_raw, target_year_val=4, n_runs=30)
        
    except Exception as e:
        print(f"Error: {e}")
        print("Running safe mock benchmark...")
        X_mock = np.random.rand(500, 200)
        X_mock = normalize(X_mock + 1e-8, norm='l1', axis=1)
        y_cls_mock = np.random.randint(0, 2, 500)
        y_year_mock = np.random.choice([2020, 2021, 2022, 2023, 2024], 500)
        run_sota_benchmark(X_mock, y_cls_mock, y_year_mock, target_year_val=2024, n_runs=2)

Loading Data...
Starting UNIFIED MOEA Benchmark: 30 Runs

>>> Starting Run 1/30...
  [Exp1_Random_Split - Run 1] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...
  [Exp2_Temporal_Transfer - Run 1] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...

>>> Starting Run 2/30...
  [Exp1_Random_Split - Run 2] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...


In [3]:
# ==============================================================
# 纯净版 Baseline 实验：双实验设置 (Random Split vs. Temporal Transfer)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time

# --- Sklearn ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # 2019年提出的自适应几何多目标算法 (Modern SOTA)
from pymoo.algorithms.moo.sms import SMSEMOA     # 极强的基于超体积(Hypervolume)的 SOTA 基线

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration & Utilities
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (基于随机森林特征重要性)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Class_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_class'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Min_Year_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return float(np.mean(opts['imp_year'][sel])) + _size_penalty(len(sel), len(x))

# ==============================================================
# 2. Pymoo Problem Definition
# ==============================================================
class FeatureSelectionProblem(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Class_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Min_Year_Relevance_Fast(mask, self.opts)
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 3. Core Evaluation Routine (7 种纯血算法大一统)
# ==============================================================
def run_moeas_and_evaluate(xtrain, xtest, ytrain, ytest, yyear_tr, run_idx, exp_name):
    print(f"  [{exp_name} - Run {run_idx}] Pre-computing Feature Importances (Random Forest)...")
    
    rf_class = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf_class.fit(xtrain, ytrain)
    imp_class = rf_class.feature_importances_
    
    rf_year = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rf_year.fit(xtrain, yyear_tr)
    imp_year = rf_year.feature_importances_
    
    opts = {'xt_in': xtrain, 'imp_class': imp_class, 'imp_year': imp_year, 'ytrain': ytrain}
    problem = FeatureSelectionProblem(opts)
    
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
    pop_size = len(ref_dirs) # 91
    
    # 统一的基础算子
    crossover = SinglePointCrossover(prob=0.8)
    mutation = BitflipMutation(prob=0.01)
    std_sampling = BinaryRandomSampling()
    
    # 7 种原装算法配置
    algorithms = {
        "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
    }
    
    def get_top_k(scores, k):
        idx = np.argsort(scores)[::-1][:k]
        return idx if len(idx) > 0 else np.array([0])
        
    clf = LinearDiscriminantAnalysis()
    results = []
    
    for algo_name, algo in algorithms.items():
        print(f"    -> Optimizing with {algo_name}...")
        
        start_time = time.time()
        res = minimize(problem, algo, ('n_gen', 40), seed=42+run_idx, verbose=False)
        elapsed_time = time.time() - start_time
        
        final_pop = res.pop.get("X").astype(int)
        scores_algo = final_pop.mean(axis=0)
        
        for k in range(1, 16):
            idx = get_top_k(scores_algo, k)
            clf.fit(xtrain[:, idx], ytrain)
            acc = clf.score(xtest[:, idx], ytest)
            
            results.append({
                "Experiment": exp_name,
                "Run": run_idx, 
                "Method": algo_name, 
                "K": k, 
                "Accuracy": acc,
                "Time_Seconds": elapsed_time
            })
            
    return results

# ==============================================================
# 4. Dual-Experiment Orchestrator
# ==============================================================
def run_sota_benchmark(features, class_labels, year_raw, target_year_val=4, n_runs=10):
    le_year = LabelEncoder()
    y_year_encoded = le_year.fit_transform(year_raw)
    
    if target_year_val in year_raw:
        target_year_idx = le_year.transform([target_year_val])[0]
    else:
        target_year_idx = np.max(y_year_encoded)

    all_results = []
    print("="*70)
    print(f"Starting UNIFIED PURE MOEA Benchmark: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Starting Run {run_idx}/{n_runs}...")
        
        # --- Exp 1: Random Split ---
        xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, _ = train_test_split(
            features, class_labels, y_year_encoded, 
            test_size=0.3, stratify=class_labels, random_state=current_seed
        )
        res1 = run_moeas_and_evaluate(xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, run_idx, "Exp1_Random_Split")
        all_results.extend(res1)
        
        # --- Exp 2: Temporal Transfer ---
        test_mask = (y_year_encoded == target_year_idx)
        train_mask = ~test_mask
        xt_tr2, xt_te2 = features[train_mask], features[test_mask]
        yt_tr2, yt_te2 = class_labels[train_mask], class_labels[test_mask]
        yr_tr2 = y_year_encoded[train_mask]
        
        res2 = run_moeas_and_evaluate(xt_tr2, xt_te2, yt_tr2, yt_te2, yr_tr2, run_idx, "Exp2_Temporal_Transfer")
        all_results.extend(res2)

    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("             PURE NATIVE MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"
    
    # ----------------------------------------------------
    # 表 1. 分类精确度 (Experiment 1: Random Split)
    # ----------------------------------------------------
    df_exp1 = df_results[df_results["Experiment"] == "Exp1_Random_Split"]
    table_acc_exp1 = df_exp1.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp1 = table_acc_exp1[[m for m in METHOD_ORDER if m in table_acc_exp1.columns]]
    
    print("\n[TABLE 1] Exp 1 (Random Split) - Classification Accuracy")
    print(table_acc_exp1)
    table_acc_exp1.to_csv("Pure_MOEA_Exp1_Accuracy.csv") 
    
    # ----------------------------------------------------
    # 表 2. 分类精确度 (Experiment 2: Temporal Transfer)
    # ----------------------------------------------------
    df_exp2 = df_results[df_results["Experiment"] == "Exp2_Temporal_Transfer"]
    table_acc_exp2 = df_exp2.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp2 = table_acc_exp2[[m for m in METHOD_ORDER if m in table_acc_exp2.columns]]
    
    print("\n[TABLE 2] Exp 2 (Temporal Transfer) - Classification Accuracy")
    print(table_acc_exp2)
    table_acc_exp2.to_csv("Pure_MOEA_Exp2_Accuracy.csv")

    # ----------------------------------------------------
    # 表 3. 计算耗时对比 (Time_Seconds)
    # ----------------------------------------------------
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Pure_MOEA_Execution_Time.csv")

    print("\n-> 纯血 7 种原生 SOTA 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/Spectral_DataSet(1).xlsx")
        
        class_col_idx = -4
        year_col_idx = -1
        y_class_raw = df.iloc[:, class_col_idx].values
        mask_cls = np.isin(y_class_raw, [2, 3])
        df_filtered = df[mask_cls].copy()
        
        balanced_indices = []
        curr_years = df_filtered.iloc[:, year_col_idx].values
        curr_classes = df_filtered.iloc[:, class_col_idx].values
        unique_years = np.unique(curr_years)
        
        for yr in unique_years:
            for cls in [2, 3]:
                match_mask = (curr_years == yr) & (curr_classes == cls)
                match_indices = np.where(match_mask)[0]
                if len(match_indices) >= 100:
                    selected = np.random.choice(match_indices, 100, replace=False)
                else: 
                    selected = match_indices
                balanced_indices.extend(selected)
                
        df_balanced = df_filtered.iloc[balanced_indices].copy()
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X = df_balanced.iloc[:, :600].values.astype(float)
        X = normalize(X + 1e-8, norm='l1', axis=1)
        y_cls = np.where(df_balanced.iloc[:, class_col_idx].values == 2, 0, 1)
        y_year_raw = df_balanced.iloc[:, year_col_idx].values
        
        run_sota_benchmark(X, y_cls, y_year_raw, target_year_val=4, n_runs=30)
        
    except Exception as e:
        print(f"Error: {e}")

Loading Data...
Starting UNIFIED PURE MOEA Benchmark: 30 Runs

>>> Starting Run 1/30...
  [Exp1_Random_Split - Run 1] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...
  [Exp2_Temporal_Transfer - Run 1] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...

>>> Starting Run 2/30...
  [Exp1_Random_Split - Run 2] Pre-computing Feature Importances (Random Forest)...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D.

In [4]:
# ==============================================================
# 纯净版 Baseline 实验：双实验设置 (Random Split vs. Temporal Transfer)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["NSGA-II", "NSGA-III", "RVEA", "SPEA2", "MOEA/D", "AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration & Utilities
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息评估特征)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Class_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_class'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Min_Year_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return float(np.mean(opts['imp_year'][sel])) + _size_penalty(len(sel), len(x))

# ==============================================================
# 2. Pymoo Problem Definition
# ==============================================================
class FeatureSelectionProblem(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：最大化分类互信息, 最小化冗余, 最小化年份互信息
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Class_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Min_Year_Relevance_Fast(mask, self.opts)
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 3. Core Evaluation Routine (7 种纯血算法大一统)
# ==============================================================
def run_moeas_and_evaluate(xtrain, xtest, ytrain, ytest, yyear_tr, run_idx, exp_name):
    current_seed = 42 + run_idx
    print(f"  [{exp_name} - Run {run_idx}] Pre-computing Mutual Information...")
    
    # --- 核心修正：统一使用 mutual_info_classif 保持公平对比 ---
    imp_class = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
    imp_year = mutual_info_classif(xtrain, yyear_tr, random_state=current_seed)
    
    opts = {'xt_in': xtrain, 'imp_class': imp_class, 'imp_year': imp_year, 'ytrain': ytrain}
    problem = FeatureSelectionProblem(opts)
    
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
    pop_size = len(ref_dirs) # 91
    
    # 统一的基础算子
    crossover = SinglePointCrossover(prob=0.8)
    mutation = BitflipMutation(prob=0.01)
    std_sampling = BinaryRandomSampling()
    
    # 7 种原装算法配置
    algorithms = {
        "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
    }
    
    def get_top_k(scores, k):
        idx = np.argsort(scores)[::-1][:k]
        return idx if len(idx) > 0 else np.array([0])
        
    clf = LinearDiscriminantAnalysis()
    results = []
    
    for algo_name, algo in algorithms.items():
        print(f"    -> Optimizing with {algo_name}...")
        
        start_time = time.time()
        res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
        elapsed_time = time.time() - start_time
        
        final_pop = res.pop.get("X").astype(int)
        scores_algo = final_pop.mean(axis=0)
        
        for k in range(1, 16):
            idx = get_top_k(scores_algo, k)
            clf.fit(xtrain[:, idx], ytrain)
            acc = clf.score(xtest[:, idx], ytest)
            
            results.append({
                "Experiment": exp_name,
                "Run": run_idx, 
                "Method": algo_name, 
                "K": k, 
                "Accuracy": acc,
                "Time_Seconds": elapsed_time
            })
            
    return results

# ==============================================================
# 4. Dual-Experiment Orchestrator
# ==============================================================
def run_sota_benchmark(features, class_labels, year_raw, target_year_val=4, n_runs=10):
    le_year = LabelEncoder()
    y_year_encoded = le_year.fit_transform(year_raw)
    
    if target_year_val in year_raw:
        target_year_idx = le_year.transform([target_year_val])[0]
    else:
        target_year_idx = np.max(y_year_encoded)

    all_results = []
    print("="*70)
    print(f"Starting UNIFIED PURE MOEA Benchmark: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Starting Run {run_idx}/{n_runs}...")
        
        # --- Exp 1: Random Split ---
        xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, _ = train_test_split(
            features, class_labels, y_year_encoded, 
            test_size=0.3, stratify=class_labels, random_state=current_seed
        )
        res1 = run_moeas_and_evaluate(xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, run_idx, "Exp1_Random_Split")
        all_results.extend(res1)
        
        # --- Exp 2: Temporal Transfer ---
        test_mask = (y_year_encoded == target_year_idx)
        train_mask = ~test_mask
        xt_tr2, xt_te2 = features[train_mask], features[test_mask]
        yt_tr2, yt_te2 = class_labels[train_mask], class_labels[test_mask]
        yr_tr2 = y_year_encoded[train_mask]
        
        res2 = run_moeas_and_evaluate(xt_tr2, xt_te2, yt_tr2, yt_te2, yr_tr2, run_idx, "Exp2_Temporal_Transfer")
        all_results.extend(res2)

    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("             PURE NATIVE MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"
    
    # ----------------------------------------------------
    # 表 1. 分类精确度 (Experiment 1: Random Split)
    # ----------------------------------------------------
    df_exp1 = df_results[df_results["Experiment"] == "Exp1_Random_Split"]
    table_acc_exp1 = df_exp1.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp1 = table_acc_exp1[[m for m in METHOD_ORDER if m in table_acc_exp1.columns]]
    
    print("\n[TABLE 1] Exp 1 (Random Split) - Classification Accuracy")
    print(table_acc_exp1)
    table_acc_exp1.to_csv("Pure_MOEA_Exp1_Accuracy.csv") 
    
    # ----------------------------------------------------
    # 表 2. 分类精确度 (Experiment 2: Temporal Transfer)
    # ----------------------------------------------------
    df_exp2 = df_results[df_results["Experiment"] == "Exp2_Temporal_Transfer"]
    table_acc_exp2 = df_exp2.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp2 = table_acc_exp2[[m for m in METHOD_ORDER if m in table_acc_exp2.columns]]
    
    print("\n[TABLE 2] Exp 2 (Temporal Transfer) - Classification Accuracy")
    print(table_acc_exp2)
    table_acc_exp2.to_csv("Pure_MOEA_Exp2_Accuracy.csv")

    # ----------------------------------------------------
    # 表 3. 计算耗时对比 (Time_Seconds)
    # ----------------------------------------------------
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Pure_MOEA_Execution_Time.csv")

    print("\n-> 纯血 7 种原生 SOTA 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/Spectral_DataSet(1).xlsx")
        
        class_col_idx = -4
        year_col_idx = -1
        y_class_raw = df.iloc[:, class_col_idx].values
        mask_cls = np.isin(y_class_raw, [2, 3])
        df_filtered = df[mask_cls].copy()
        
        balanced_indices = []
        curr_years = df_filtered.iloc[:, year_col_idx].values
        curr_classes = df_filtered.iloc[:, class_col_idx].values
        unique_years = np.unique(curr_years)
        
        for yr in unique_years:
            for cls in [2, 3]:
                match_mask = (curr_years == yr) & (curr_classes == cls)
                match_indices = np.where(match_mask)[0]
                if len(match_indices) >= 100:
                    selected = np.random.choice(match_indices, 100, replace=False)
                else: 
                    selected = match_indices
                balanced_indices.extend(selected)
                
        df_balanced = df_filtered.iloc[balanced_indices].copy()
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X = df_balanced.iloc[:, :600].values.astype(float)
        X = normalize(X + 1e-8, norm='l1', axis=1)
        y_cls = np.where(df_balanced.iloc[:, class_col_idx].values == 2, 0, 1)
        y_year_raw = df_balanced.iloc[:, year_col_idx].values
        
        run_sota_benchmark(X, y_cls, y_year_raw, target_year_val=4, n_runs=30)
        
    except Exception as e:
        print(f"Error: {e}")

Loading Data...
Starting UNIFIED PURE MOEA Benchmark: 30 Runs

>>> Starting Run 1/30...
  [Exp1_Random_Split - Run 1] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...
  [Exp2_Temporal_Transfer - Run 1] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...

>>> Starting Run 2/30...
  [Exp1_Random_Split - Run 2] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimi

In [6]:
# ==============================================================
# 纯净版 Baseline 实验：双实验设置 (Random Split vs. Temporal Transfer)
# 综合对比测试：7 种纯正血统的 SOTA MOEAs (原生 PyMoo 实现，绝无魔改)
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) 原生纯血算法 ---
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.spea2 import SPEA2
from pymoo.algorithms.moo.rvea import RVEA
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.age import AGEMOEA     # Modern SOTA
from pymoo.algorithms.moo.sms import SMSEMOA     # Hypervolume SOTA

from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序
METHOD_ORDER = ["AGE-MOEA", "SMS-EMOA"]

# ==============================================================
# 0. Configuration & Utilities
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息评估特征)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Class_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_class'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Min_Year_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return float(np.mean(opts['imp_year'][sel])) + _size_penalty(len(sel), len(x))

# ==============================================================
# 2. Pymoo Problem Definition
# ==============================================================
class FeatureSelectionProblem(Problem):
    def __init__(self, opts, **kwargs):
        # 3 个目标：最大化分类互信息, 最小化冗余, 最小化年份互信息
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2, f3 = np.zeros(X.shape[0]), np.zeros(X.shape[0]), np.zeros(X.shape[0])
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Class_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Min_Year_Relevance_Fast(mask, self.opts)
        out["F"] = np.column_stack([f1, f2, f3])


# ==============================================================
# 3. Core Evaluation Routine (7 种纯血算法大一统)
# ==============================================================
def run_moeas_and_evaluate(xtrain, xtest, ytrain, ytest, yyear_tr, run_idx, exp_name):
    current_seed = 42 + run_idx
    print(f"  [{exp_name} - Run {run_idx}] Pre-computing Mutual Information...")
    
    # --- 核心修正：统一使用 mutual_info_classif 保持公平对比 ---
    imp_class = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
    imp_year = mutual_info_classif(xtrain, yyear_tr, random_state=current_seed)
    
    opts = {'xt_in': xtrain, 'imp_class': imp_class, 'imp_year': imp_year, 'ytrain': ytrain}
    problem = FeatureSelectionProblem(opts)
    
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
    pop_size = len(ref_dirs) # 91
    
    # 统一的基础算子
    crossover = SinglePointCrossover(prob=0.8)
    mutation = BitflipMutation(prob=0.01)
    std_sampling = BinaryRandomSampling()
    
    # 7 种原装算法配置
    algorithms = {
        "NSGA-II": NSGA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "NSGA-III": NSGA3(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SPEA2": SPEA2(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "RVEA": RVEA(ref_dirs=ref_dirs, pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "MOEA/D": MOEAD(ref_dirs=ref_dirs, n_neighbors=15, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "AGE-MOEA": AGEMOEA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation),
        "SMS-EMOA": SMSEMOA(pop_size=pop_size, sampling=std_sampling, crossover=crossover, mutation=mutation)
    }
    
    def get_top_k(scores, k):
        idx = np.argsort(scores)[::-1][:k]
        return idx if len(idx) > 0 else np.array([0])
        
    clf = LinearDiscriminantAnalysis()
    results = []
    
    for algo_name, algo in algorithms.items():
        print(f"    -> Optimizing with {algo_name}...")
        
        start_time = time.time()
        res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
        elapsed_time = time.time() - start_time
        
        final_pop = res.pop.get("X").astype(int)
        scores_algo = final_pop.mean(axis=0)
        
        for k in range(1, 16):
            idx = get_top_k(scores_algo, k)
            clf.fit(xtrain[:, idx], ytrain)
            acc = clf.score(xtest[:, idx], ytest)
            
            results.append({
                "Experiment": exp_name,
                "Run": run_idx, 
                "Method": algo_name, 
                "K": k, 
                "Accuracy": acc,
                "Time_Seconds": elapsed_time
            })
            
    return results

# ==============================================================
# 4. Dual-Experiment Orchestrator
# ==============================================================
def run_sota_benchmark(features, class_labels, year_raw, target_year_val=4, n_runs=10):
    le_year = LabelEncoder()
    y_year_encoded = le_year.fit_transform(year_raw)
    
    if target_year_val in year_raw:
        target_year_idx = le_year.transform([target_year_val])[0]
    else:
        target_year_idx = np.max(y_year_encoded)

    all_results = []
    print("="*70)
    print(f"Starting UNIFIED PURE MOEA Benchmark: {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Starting Run {run_idx}/{n_runs}...")
        
        # --- Exp 1: Random Split ---
        xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, _ = train_test_split(
            features, class_labels, y_year_encoded, 
            test_size=0.3, stratify=class_labels, random_state=current_seed
        )
        res1 = run_moeas_and_evaluate(xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, run_idx, "Exp1_Random_Split")
        all_results.extend(res1)
        
        # --- Exp 2: Temporal Transfer ---
        test_mask = (y_year_encoded == target_year_idx)
        train_mask = ~test_mask
        xt_tr2, xt_te2 = features[train_mask], features[test_mask]
        yt_tr2, yt_te2 = class_labels[train_mask], class_labels[test_mask]
        yr_tr2 = y_year_encoded[train_mask]
        
        res2 = run_moeas_and_evaluate(xt_tr2, xt_te2, yt_tr2, yt_te2, yr_tr2, run_idx, "Exp2_Temporal_Transfer")
        all_results.extend(res2)

    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("             PURE NATIVE MOEA TABLES (Mean ± Std)")
    print("="*80)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"
    
    # ----------------------------------------------------
    # 表 1. 分类精确度 (Experiment 1: Random Split)
    # ----------------------------------------------------
    df_exp1 = df_results[df_results["Experiment"] == "Exp1_Random_Split"]
    table_acc_exp1 = df_exp1.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp1 = table_acc_exp1[[m for m in METHOD_ORDER if m in table_acc_exp1.columns]]
    
    print("\n[TABLE 1] Exp 1 (Random Split) - Classification Accuracy")
    print(table_acc_exp1)
    table_acc_exp1.to_csv("Pure_MOEA_Exp1_Accuracy.csv") 
    
    # ----------------------------------------------------
    # 表 2. 分类精确度 (Experiment 2: Temporal Transfer)
    # ----------------------------------------------------
    df_exp2 = df_results[df_results["Experiment"] == "Exp2_Temporal_Transfer"]
    table_acc_exp2 = df_exp2.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp2 = table_acc_exp2[[m for m in METHOD_ORDER if m in table_acc_exp2.columns]]
    
    print("\n[TABLE 2] Exp 2 (Temporal Transfer) - Classification Accuracy")
    print(table_acc_exp2)
    table_acc_exp2.to_csv("Pure_MOEA_Exp2_Accuracy.csv")

    # ----------------------------------------------------
    # 表 3. 计算耗时对比 (Time_Seconds)
    # ----------------------------------------------------
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Pure_MOEA_Execution_Time.csv")

    print("\n-> 纯血 7 种原生 SOTA 算法基线对比及运行时间 CSV 文件生成完毕！")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/Spectral_DataSet(1).xlsx")
        
        class_col_idx = -4
        year_col_idx = -1
        y_class_raw = df.iloc[:, class_col_idx].values
        mask_cls = np.isin(y_class_raw, [2, 3])
        df_filtered = df[mask_cls].copy()
        
        balanced_indices = []
        curr_years = df_filtered.iloc[:, year_col_idx].values
        curr_classes = df_filtered.iloc[:, class_col_idx].values
        unique_years = np.unique(curr_years)
        
        for yr in unique_years:
            for cls in [2, 3]:
                match_mask = (curr_years == yr) & (curr_classes == cls)
                match_indices = np.where(match_mask)[0]
                if len(match_indices) >= 100:
                    selected = np.random.choice(match_indices, 100, replace=False)
                else: 
                    selected = match_indices
                balanced_indices.extend(selected)
                
        df_balanced = df_filtered.iloc[balanced_indices].copy()
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X = df_balanced.iloc[:, :600].values.astype(float)
        X = normalize(X + 1e-8, norm='l1', axis=1)
        y_cls = np.where(df_balanced.iloc[:, class_col_idx].values == 2, 0, 1)
        y_year_raw = df_balanced.iloc[:, year_col_idx].values
        
        run_sota_benchmark(X, y_cls, y_year_raw, target_year_val=4, n_runs=30)
        
    except Exception as e:
        print(f"Error: {e}")

Loading Data...
Starting UNIFIED PURE MOEA Benchmark: 30 Runs

>>> Starting Run 1/30...
  [Exp1_Random_Split - Run 1] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...
  [Exp2_Temporal_Transfer - Run 1] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimizing with SMS-EMOA...

>>> Starting Run 2/30...
  [Exp1_Random_Split - Run 2] Pre-computing Mutual Information...
    -> Optimizing with NSGA-II...
    -> Optimizing with NSGA-III...
    -> Optimizing with SPEA2...
    -> Optimizing with RVEA...
    -> Optimizing with MOEA/D...
    -> Optimizing with AGE-MOEA...
    -> Optimi

In [5]:
# ==============================================================
# 纯净版 Baseline 实验：双实验设置 (Random Split vs. Temporal Transfer)
# 专项测试：仅运行 MOFS-REPLS 和 DREA-FS
# 核心修正：严格对齐互信息 (Mutual Information) 作为特征评价指标
# 依赖库要求：pip install pymoo==0.6.0 scikit-learn pandas numpy
# ==============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import time
from scipy import stats

# --- Sklearn ---
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif  # 核心修正：引入互信息

# --- Pymoo (v0.6.0+) ---
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.pntx import SinglePointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

# 统一的方法排列顺序（仅保留定制算法）
METHOD_ORDER = ["MOFS-REPLS", "DREA-FS"]

# ==============================================================
# 0. Configuration & Utilities
# ==============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ==============================================================
# 1. Safe & Fast Fitness Functions (统一使用互信息评估特征)
# ==============================================================
def _size_penalty(num_selected, total_features, lam=0.01):
    if num_selected == 0: return 1.0
    return lam * (num_selected / max(1, total_features))

def Obj1_Max_Class_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return -float(np.mean(opts['imp_class'][sel])) + _size_penalty(len(sel), len(x))

def Obj2_Min_Redundancy_Safe(x, opts):
    sel = np.where(x == 1)[0]
    xt_in = opts['xt_in']
    if len(sel) <= 1: return 0.0 + _size_penalty(len(sel), xt_in.shape[1])
    Xs = xt_in[:, sel]
    Xs = (Xs - Xs.mean(axis=0)) / (Xs.std(axis=0) + 1e-8)
    C = np.abs(np.dot(Xs.T, Xs) / max(1, Xs.shape[0] - 1))
    np.fill_diagonal(C, 0)
    n = C.shape[0]
    return float(C.sum() / (n * (n - 1))) + _size_penalty(n, xt_in.shape[1])

def Obj3_Min_Year_Relevance_Fast(x, opts):
    sel = np.where(x == 1)[0]
    if len(sel) == 0: return 1e6
    return float(np.mean(opts['imp_year'][sel])) + _size_penalty(len(sel), len(x))

# ==============================================================
# 2. Pymoo Problem Definition
# ==============================================================
class FeatureSelectionProblem(Problem):
    def __init__(self, opts, **kwargs):
        super().__init__(n_var=opts['xt_in'].shape[1], 
                         n_obj=3, 
                         n_ieq_constr=0, 
                         xl=0, xu=1, vtype=bool, **kwargs)
        self.opts = opts

    def _evaluate(self, X, out, *args, **kwargs):
        f1 = np.zeros(X.shape[0])
        f2 = np.zeros(X.shape[0])
        f3 = np.zeros(X.shape[0])
        
        for i in range(X.shape[0]):
            mask = X[i].astype(int)
            f1[i] = Obj1_Max_Class_Relevance_Fast(mask, self.opts)
            f2[i] = Obj2_Min_Redundancy_Safe(mask, self.opts)
            f3[i] = Obj3_Min_Year_Relevance_Fast(mask, self.opts)
            
        out["F"] = np.column_stack([f1, f2, f3])

# ==============================================================
# 3. 算法核心算子定义 (Samplings & Algorithms)
# ==============================================================
class REPLSSampling(Sampling):
    """针对 MOFS-REPLS 算法定制的偏好注入采样器"""
    def _do(self, problem, n_samples, **kwargs):
        D = problem.n_var
        X_init = np.zeros((n_samples, D), dtype=bool)
        weights = problem.opts['imp_class']
        weights = np.maximum(weights, 1e-8)
        probs = weights / np.sum(weights)
        
        for i in range(n_samples):
            interval = D / (3 * n_samples)
            sel_num = int(1 + np.ceil(interval * (i + 1)))
            sel_num = min(sel_num, D)
            chosen_features = np.random.choice(D, sel_num, replace=False, p=probs)
            X_init[i, chosen_features] = True
            
        return X_init

class MOFS_REPLS(NSGA2):
    """MOFS-REPLS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.me_archive = {} 
        self.nd_archive = []
        self.P = 5 

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        for ind in self.pop:
            x = ind.get("X")
            f_err = ind.get("F")[0]
            num_f = np.sum(x)
            if num_f not in self.me_archive or f_err < self.me_archive[num_f].get("F")[0]:
                self.me_archive[num_f] = ind
                
        fronts = self.pop.get("rank")
        if fronts is not None:
            nd_indices = np.where(fronts == 0)[0]
            self.nd_archive = self.pop[nd_indices].copy()

    def _infill(self):
        infills = super()._infill()
        if len(self.nd_archive) >= self.P:
            X_inf = infills.get("X")
            for i in range(len(X_inf)):
                ind_float = X_inf[i].astype(float)
                dists = [np.linalg.norm(ind_float - nd.get("X").astype(float)) for nd in self.nd_archive]
                p_nearest_indices = np.argsort(dists)[:self.P]
                
                leader = self.nd_archive[np.random.choice(p_nearest_indices)]
                leader_features = np.where(leader.get("X"))[0]
                
                rand_val = np.random.rand()
                if rand_val < 0.33 and len(leader_features) > 0: 
                    add_f = np.random.choice(leader_features, max(1, len(leader_features)//2), replace=False)
                    X_inf[i, add_f] = True
                elif rand_val < 0.66: 
                    current_features = np.where(X_inf[i])[0]
                    if len(current_features) > 1:
                        del_f = np.random.choice(current_features, max(1, len(current_features)//4), replace=False)
                        X_inf[i, del_f] = False
            infills.set("X", X_inf)
        return infills


class DREAFSSampling(Sampling):
    """针对 DREA-FS 算法定制的双视角降维采样器"""
    def _do(self, problem, n_samples, **kwargs):
        X_data = problem.opts['xt_in']
        imp_class = problem.opts['imp_class']
        n_features = X_data.shape[1]
        
        X_init = np.zeros((n_samples, n_features), dtype=bool)
        half_pop = n_samples // 2
        
        knee_point = np.median(imp_class[imp_class > 0]) if np.any(imp_class > 0) else 0
        important_features = np.where(imp_class >= knee_point)[0]
        
        for i in range(half_pop):
            if i % 2 == 0:
                mask = np.random.choice([True, False], size=len(important_features))
            else:
                mask = ~mask
            if len(important_features) > 0:
                X_init[i, important_features] = mask
                
        m_groups = max(2, int(5 * np.log2(n_features) if n_features > 0 else 2))
        m_groups = min(m_groups, n_features)
        
        kmeans = KMeans(n_clusters=m_groups, random_state=42, n_init=1)
        labels = kmeans.fit_predict(X_data.T)
        
        for i in range(half_pop, n_samples):
            selected_groups = np.random.choice(m_groups, m_groups // 2, replace=False)
            for g in selected_groups:
                features_in_g = np.where(labels == g)[0]
                if len(features_in_g) > 0:
                    chosen_f = np.random.choice(features_in_g, max(1, len(features_in_g)//2), replace=False)
                    X_init[i, chosen_f] = True
        return X_init

class DREA_FS(NSGA2):
    """DREA-FS 完整生命周期类"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.archive_c = []  
        
    def _infill(self):
        infills = super()._infill()
        if len(self.archive_c) > 0 and self.n_gen % 5 == 0:
            replace_idx = np.random.choice(len(infills), min(5, len(infills)), replace=False)
            X_inf = infills.get("X")
            for idx in replace_idx:
                X_inf[idx] = self.archive_c[np.random.randint(len(self.archive_c))]
            infills.set("X", X_inf)
        return infills

    def _advance(self, infills=None, **kwargs):
        super()._advance(infills, **kwargs)
        F = self.pop.get("F")
        X = self.pop.get("X")
        elite_indices = np.argsort(F[:, 0])[:10]
        self.archive_c.extend(X[elite_indices])
        if len(self.archive_c) > 50:
            self.archive_c = self.archive_c[-50:]

# ==============================================================
# 4. Core Evaluation Routine (仅运行两个定制 MOEAs)
# ==============================================================
def run_moeas_and_evaluate(xtrain, xtest, ytrain, ytest, yyear_tr, run_idx, exp_name):
    current_seed = 42 + run_idx
    print(f"  [{exp_name} - Run {run_idx}] Pre-computing Mutual Information...")
    
    # --- 核心修正：使用互信息代替随机森林，保持对齐 ---
    imp_class = mutual_info_classif(xtrain, ytrain, random_state=current_seed)
    imp_year = mutual_info_classif(xtrain, yyear_tr, random_state=current_seed)
    
    opts = {'xt_in': xtrain, 'imp_class': imp_class, 'imp_year': imp_year, 'ytrain': ytrain}
    problem = FeatureSelectionProblem(opts)
    
    ref_dirs = get_reference_directions("das-dennis", 3, n_partitions=12)
    pop_size = len(ref_dirs) # 91
    
    # 统一算子
    crossover = SinglePointCrossover(prob=0.8)
    mutation = BitflipMutation(prob=0.01)
    
    # 仅加载 2 种定制算法
    algorithms = {
        "MOFS-REPLS": MOFS_REPLS(pop_size=pop_size, sampling=REPLSSampling(), crossover=crossover, mutation=mutation),
        "DREA-FS": DREA_FS(pop_size=pop_size, sampling=DREAFSSampling(), crossover=crossover, mutation=mutation)
    }
    
    def get_top_k(scores, k):
        idx = np.argsort(scores)[::-1][:k]
        return idx if len(idx) > 0 else np.array([0])
        
    clf = LinearDiscriminantAnalysis()
    results = []
    
    for algo_name, algo in algorithms.items():
        print(f"    -> Optimizing with {algo_name}...")
        
        start_time = time.time()
        res = minimize(problem, algo, ('n_gen', 40), seed=current_seed, verbose=False)
        elapsed_time = time.time() - start_time
        
        final_pop = res.pop.get("X").astype(int)
        scores_algo = final_pop.mean(axis=0)
        
        for k in range(1, 16):
            idx = get_top_k(scores_algo, k)
            clf.fit(xtrain[:, idx], ytrain)
            acc = clf.score(xtest[:, idx], ytest)
            
            results.append({
                "Experiment": exp_name,
                "Run": run_idx, 
                "Method": algo_name, 
                "K": k, 
                "Accuracy": acc,
                "Time_Seconds": elapsed_time
            })
            
    return results

# ==============================================================
# 5. Dual-Experiment Orchestrator
# ==============================================================
def run_sota_benchmark(features, class_labels, year_raw, target_year_val=4, n_runs=10):
    le_year = LabelEncoder()
    y_year_encoded = le_year.fit_transform(year_raw)
    
    if target_year_val in year_raw:
        target_year_idx = le_year.transform([target_year_val])[0]
    else:
        target_year_idx = np.max(y_year_encoded)

    all_results = []
    print("="*70)
    print(f"Starting CUSTOM MOEAs Benchmark (MI Aligned): {n_runs} Runs")
    print("="*70)
    
    for run_idx in range(1, n_runs + 1):
        current_seed = 42 + run_idx
        set_seed(current_seed)
        print(f"\n>>> Starting Run {run_idx}/{n_runs}...")
        
        # --- Exp 1: Random Split ---
        xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, _ = train_test_split(
            features, class_labels, y_year_encoded, 
            test_size=0.3, stratify=class_labels, random_state=current_seed
        )
        res1 = run_moeas_and_evaluate(xt_tr1, xt_te1, yt_tr1, yt_te1, yr_tr1, run_idx, "Exp1_Random_Split")
        all_results.extend(res1)
        
        # --- Exp 2: Temporal Transfer ---
        test_mask = (y_year_encoded == target_year_idx)
        train_mask = ~test_mask
        xt_tr2, xt_te2 = features[train_mask], features[test_mask]
        yt_tr2, yt_te2 = class_labels[train_mask], class_labels[test_mask]
        yr_tr2 = y_year_encoded[train_mask]
        
        res2 = run_moeas_and_evaluate(xt_tr2, xt_te2, yt_tr2, yt_te2, yr_tr2, run_idx, "Exp2_Temporal_Transfer")
        all_results.extend(res2)

    df_results = pd.DataFrame(all_results)
    
    print("\n\n" + "="*80)
    print("             CUSTOM MOEAs TABLES (Mean ± Std)")
    print("="*80)
    
    def calc_mean_std(x):
        return f"{x.mean():.4f} ± {x.std(ddof=1):.4f}" if len(x) > 1 else f"{x.mean():.4f} ± 0.0000"
    
    # ----------------------------------------------------
    # 表 1. 分类精确度 (Experiment 1: Random Split)
    # ----------------------------------------------------
    df_exp1 = df_results[df_results["Experiment"] == "Exp1_Random_Split"]
    table_acc_exp1 = df_exp1.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp1 = table_acc_exp1[[m for m in METHOD_ORDER if m in table_acc_exp1.columns]]
    
    print("\n[TABLE 1] Exp 1 (Random Split) - Classification Accuracy")
    print(table_acc_exp1)
    table_acc_exp1.to_csv("Custom_MOEAs_Exp1_Accuracy.csv") 
    
    # ----------------------------------------------------
    # 表 2. 分类精确度 (Experiment 2: Temporal Transfer)
    # ----------------------------------------------------
    df_exp2 = df_results[df_results["Experiment"] == "Exp2_Temporal_Transfer"]
    table_acc_exp2 = df_exp2.pivot_table(index="K", columns="Method", values="Accuracy", aggfunc=calc_mean_std)
    table_acc_exp2 = table_acc_exp2[[m for m in METHOD_ORDER if m in table_acc_exp2.columns]]
    
    print("\n[TABLE 2] Exp 2 (Temporal Transfer) - Classification Accuracy")
    print(table_acc_exp2)
    table_acc_exp2.to_csv("Custom_MOEAs_Exp2_Accuracy.csv")

    # ----------------------------------------------------
    # 表 3. 计算耗时对比 (Time_Seconds)
    # ----------------------------------------------------
    table_time = df_results.pivot_table(index="Method", values="Time_Seconds", aggfunc=calc_mean_std)
    table_time = table_time.reindex(METHOD_ORDER).dropna()
    print("\n[TABLE 3] Execution Time (Seconds) per Algorithm")
    print(table_time)
    table_time.to_csv("Custom_MOEAs_Execution_Time.csv")

    print("\n-> 定制算法 MOFS-REPLS 与 DREA-FS 运行完毕！CSV 文件已生成。")

if __name__ == "__main__":
    try:
        print("Loading Data...")
        df = pd.read_excel("C:/Users/zhan0035/Downloads/Spectral_DataSet(1).xlsx")
        
        class_col_idx = -4
        year_col_idx = -1
        y_class_raw = df.iloc[:, class_col_idx].values
        mask_cls = np.isin(y_class_raw, [2, 3])
        df_filtered = df[mask_cls].copy()
        
        balanced_indices = []
        curr_years = df_filtered.iloc[:, year_col_idx].values
        curr_classes = df_filtered.iloc[:, class_col_idx].values
        unique_years = np.unique(curr_years)
        
        for yr in unique_years:
            for cls in [2, 3]:
                match_mask = (curr_years == yr) & (curr_classes == cls)
                match_indices = np.where(match_mask)[0]
                if len(match_indices) >= 100:
                    selected = np.random.choice(match_indices, 100, replace=False)
                else: 
                    selected = match_indices
                balanced_indices.extend(selected)
                
        df_balanced = df_filtered.iloc[balanced_indices].copy()
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        
        X = df_balanced.iloc[:, :600].values.astype(float)
        X = normalize(X + 1e-8, norm='l1', axis=1)
        y_cls = np.where(df_balanced.iloc[:, class_col_idx].values == 2, 0, 1)
        y_year_raw = df_balanced.iloc[:, year_col_idx].values
        
        run_sota_benchmark(X, y_cls, y_year_raw, target_year_val=4, n_runs=30)
        
    except Exception as e:
        print(f"Error: {e}")

Loading Data...
Starting CUSTOM MOEAs Benchmark (MI Aligned): 30 Runs

>>> Starting Run 1/30...
  [Exp1_Random_Split - Run 1] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...
  [Exp2_Temporal_Transfer - Run 1] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...

>>> Starting Run 2/30...
  [Exp1_Random_Split - Run 2] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...
  [Exp2_Temporal_Transfer - Run 2] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...

>>> Starting Run 3/30...
  [Exp1_Random_Split - Run 3] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...
  [Exp2_Temporal_Transfer - Run 3] Pre-computing Mutual Information...
    -> Optimizing with MOFS-REPLS...
    -> Optimizing with DREA-FS...

>>> Starting R